"""
TAL-GRN Preprocessing Pipeline
================================
Matches the preprocessing from the reference paper (UALE) for the
Benchmark Diagnostic MRI and Medical Imaging Dataset.

Steps applied:
  1. Image standardization  (resize → center-crop → normalize)
  2. Data augmentation      (Gaussian noise, color jitter, flips,
                             rotations, brightness, Mixup)
  3. Corrupted image handling (blank tensor replacement)

NOT applied (by design for TAL-GRN):
  - Class imbalance handling / oversampling
  - Expert-specific preprocessing (Gabor, Sobel, histogram equalization)

Drive layout expected:
  MyDrive/
    Benchmark Diagnostic MRI and Medical Imaging Dataset/
      Medical Imaging Dataset/
        Acute Cerebellitis in HIV/
          img1.jpg
          img2.jpg
        ...

Output saved to:
  MyDrive/TAL_GRN_Preprocessed/
    train/   <class_name>/  *.pt
    val/     <class_name>/  *.pt
    test/    <class_name>/  *.pt

Usage (in Google Colab):
  1. Mount Drive
  2. Run this script
"""

In [ ]:
# ── 0. Colab / Drive mount ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import random
import shutil
from pathlib import Path

import numpy as np
from PIL import Image, UnidentifiedImageError

import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder

In [ ]:
from pathlib import Path

# ── 2. Paths ──────────────────────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive')

RAW_ROOT     = DRIVE_ROOT / 'Benchmark Diagnostic MRI and Medical Imaging Dataset' \
                           / 'Medical Imaging Dataset'

OUTPUT_ROOT  = DRIVE_ROOT / 'TAL_GRN_Preprocessed_dataset'

# Create output directory
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Output folder created at:", OUTPUT_ROOT)

Output folder created at: /content/drive/MyDrive/TAL_GRN_Preprocessed_dataset


In [ ]:
IMAGE_SIZE   = 224

TRAIN_RATIO  = 0.68
VAL_RATIO    = 0.12

SEED         = 42
SAVE_QUALITY = 95

# Augmentation params
GAUSS_STD    = 0.01
JITTER       = 0.1
FLIP_PROB    = 0.5
ROT_DEGREES  = 10
BRIGHT_DELTA = 0.2

random.seed(SEED)
np.random.seed(SEED)

IMG_EXTENSIONS = {
    '.jpg',
    '.jpeg',
    '.png',
    '.bmp',
    '.tiff',
    '.tif'
}

In [ ]:
# Validation / Test Transform
eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
])

# Training Transform
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    T.RandomHorizontalFlip(p=FLIP_PROB),

    T.RandomRotation(
        degrees=ROT_DEGREES
    ),

    T.ColorJitter(
        brightness=BRIGHT_DELTA,
        contrast=JITTER,
        saturation=JITTER,
        hue=JITTER,
    ),
])

Gaussian Noise Function

In [ ]:
def add_gaussian_noise(img: Image.Image, std: float = GAUSS_STD) -> Image.Image:
    arr   = np.array(img).astype(np.float32) / 255.0
    noise = np.random.normal(0, std, arr.shape).astype(np.float32)
    arr   = np.clip(arr + noise, 0.0, 1.0)
    return Image.fromarray((arr * 255).astype(np.uint8))

In [ ]:
def build_splits(raw_root: Path):
    classes = sorted([
        d.name for d in raw_root.iterdir()
        if d.is_dir() and not d.name.startswith('.')
    ])
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_s, val_s, test_s = [], [], []

    print(f"\n{'Class':<50} {'Total':>6} {'Train':>6} {'Val':>5} {'Test':>5}")
    print("-" * 75)

    for cls in classes:
        cls_dir = raw_root / cls
        files   = sorted([
            p for p in cls_dir.rglob('*')
            if p.suffix.lower() in IMG_EXTENSIONS
        ])
        random.shuffle(files)

        n       = len(files)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)

        idx = class_to_idx[cls]
        train_s.extend([(str(f), idx, cls) for f in files[:n_train]])
        val_s  .extend([(str(f), idx, cls) for f in files[n_train:n_train + n_val]])
        test_s .extend([(str(f), idx, cls) for f in files[n_train + n_val:]])

        n_test = n - n_train - n_val
        print(f"  {cls:<48} {n:>6} {n_train:>6} {n_val:>5} {n_test:>5}")

    return train_s, val_s, test_s, class_to_idx

In [ ]:
def save_split(
    samples,
    split_name,
    transform,
    is_train=False
):

    split_dir = OUTPUT_ROOT / split_name

    skipped = 0
    saved   = 0
    resumed = 0

    print(
        f"\n[{split_name.upper()}]"
        f" Processing {len(samples)} images..."
    )

    # =====================================================
    # Build set of already processed files
    # =====================================================

    existing_files = set()

    if split_dir.exists():

        for cls_dir in split_dir.iterdir():

            if cls_dir.is_dir():

                for f in cls_dir.glob("*.jpg"):
                    existing_files.add(f.stem)

    print(
        f"[INFO] Found "
        f"{len(existing_files)} already processed images."
    )

    # =====================================================
    # Main loop
    # =====================================================

    for i, (path, label, cls_name) in enumerate(samples):

        stem = Path(path).stem

        # =================================================
        # Skip already processed images instantly
        # =================================================

        if stem in existing_files:
            resumed += 1
            continue

        out_dir = split_dir / cls_name

        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        out_path = out_dir / f"{stem}.jpg"

        try:

            img = Image.open(path).convert('RGB')

            # Resize + augmentation
            img = transform(img)

            # Gaussian noise for training only
            if is_train:

                img = add_gaussian_noise(
                    img,
                    GAUSS_STD
                )

            img.save(
                out_path,
                'JPEG',
                quality=SAVE_QUALITY
            )

            saved += 1

        except (
            UnidentifiedImageError,
            OSError,
            Exception
        ) as e:

            print(
                f"[WARN] Skipped corrupted image:"
                f" {path} ({e})"
            )

            skipped += 1

        # =================================================
        # Progress logging
        # =================================================

        if (i + 1) % 500 == 0:

            print(
                f"{i+1}/{len(samples)} checked | "
                f"New saved: {saved} | "
                f"Resumed: {resumed}"
            )

    print("\n===================================")

    print(f"✓ Newly Saved : {saved}")

    print(f"✓ Already Existing : {resumed}")

    print(f"✓ Corrupted Skipped : {skipped}")

    print(f"→ {split_dir}")

    print("===================================\n")

In [ ]:
import json
def main():

    print("=" * 60)
    print("TAL-GRN Image Preprocessing Pipeline")
    print("=" * 60)

    print(f"Source : {RAW_ROOT}")
    print(f"Output : {OUTPUT_ROOT}")

    print(
        f"Image Size : "
        f"{IMAGE_SIZE}x{IMAGE_SIZE}"
    )

    # STEP 1
    print(
        "\n[STEP 1]"
        " Building dataset splits..."
    )

    (
        train_s,
        val_s,
        test_s,
        class_to_idx
    ) = build_splits(RAW_ROOT)

    total = (
        len(train_s)
        + len(val_s)
        + len(test_s)
    )

    print(f"\nTotal Images : {total}")

    print(f"Train : {len(train_s)}")
    print(f"Val   : {len(val_s)}")
    print(f"Test  : {len(test_s)}")

    print(
        f"Classes : "
        f"{len(class_to_idx)}"
    )

    # STEP 2
    print(
        "\n[STEP 2]"
        " Saving processed images..."
    )

    save_split(
        train_s,
        'train',
        train_transform,
        is_train=True
    )

    save_split(
        val_s,
        'val',
        eval_transform,
        is_train=False
    )

    save_split(
        test_s,
        'test',
        eval_transform,
        is_train=False
    )

    # STEP 3
    meta = {

        'class_to_idx': class_to_idx,

        'num_classes': len(class_to_idx),

        'split_counts': {
            'train': len(train_s),
            'val': len(val_s),
            'test': len(test_s),
        },

        'config': {
            'image_size': IMAGE_SIZE,
            'gauss_std': GAUSS_STD,
            'flip_prob': FLIP_PROB,
            'rot_degrees': ROT_DEGREES,
            'brightness': BRIGHT_DELTA,
            'jitter': JITTER,
            'seed': SEED,
        }
    }

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    meta_path = (
        OUTPUT_ROOT
        / 'split_metadata.json'
    )

    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)

    print(
        f"\n[INFO] Metadata saved → "
        f"{meta_path}"
    )

    print("\n✅ PREPROCESSING COMPLETE!")

    print(
        f"\nSaved to:\n"
        f"{OUTPUT_ROOT}"
    )

if __name__ == '__main__':
    main()

TAL-GRN Image Preprocessing Pipeline
Source : /content/drive/MyDrive/Benchmark Diagnostic MRI and Medical Imaging Dataset/Medical Imaging Dataset
Output : /content/drive/MyDrive/TAL_GRN_Preprocessed_dataset
Image Size : 224x224

[STEP 1] Building dataset splits...

Class                                               Total  Train   Val  Test
---------------------------------------------------------------------------
  Acute Cerebellitis in HIV                           976    663   117   196
  Acute Unilateral Cerebellitis in HIV                488    331    58    99
  Adenomyosis in Gravid Uterus                        784    533    94   157
  Balloon Cell Cortical Dysplasia                    1424    968   170   286
  Bilateral Osgood-Schlatter Disease with Chronic Inflammatory Arthritis    656    446    78   132
  Bilateral Ulnar Impaction Syndrome                  592    402    71   119
  Carolis Disease                                    1464    995   175   294
  Congenital Toxopla